# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Samarjamal326/Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os
import subprocess
import pandas as pd
import numpy as np

repo_dir = "/content/Flyrank"
repo_url = "https://github.com/Samarjamal326/Flyrank.git"

if not os.path.exists(repo_dir):
    subprocess.run(["git", "clone", repo_url, repo_dir], check=True)

os.chdir(repo_dir)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

Dataset shape: 30,000 rows × 44 columns


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Task type: Ranking / Scoring")
print("Output: Opportunity score")
print("Decision: Which content items should be reviewed first?")

Task type: Ranking / Scoring
Output: Opportunity score
Decision: Which content items should be reviewed first?


## 2. Target or proxy

The eventual target should be an **observed future-window engagement outcome**, rather than a label created from a rule.

For this lane, I would use a future-window CTR or engagement outcome and adjust the comparison for search-position and impression volume.

The starter dataset does not contain a separate future window, so I will use its observed CTR only to understand the current data structure. The actual predictive target will be defined from a later time window when working with the warehouse data.

A possible target column is therefore:

**future_ctr** — observed CTR measured in a later evaluation window.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Sketch the target structure without pretending the starter data
# already contains a future-window target.

lane_df = df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "engagement_rate"
    ]
].copy()

lane_df["future_ctr"] = np.nan

lane_df.head()

,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,future_ctr
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,5.88,NaN
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,0.00,NaN
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,0.00,NaN
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,1.28,NaN
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,0.00,NaN


## 3. Success metric

The main success metric will be **Precision@K**.

The practical question is not whether every page receives a perfect score. It is whether the highest-ranked pages contain a useful concentration of genuine future engagement opportunities.

I will therefore evaluate how many of the top-K recommended pages meet the defined future-outcome criterion.

I will also compare the scoring approach against a simple rule-based baseline before claiming that ML adds value.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
K = 10

print(f"Primary evaluation metric: Precision@{K}")
print("Baseline comparison: simple rule-based ranking")

Primary evaluation metric: Precision@10
Baseline comparison: simple rule-based ranking


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content item**.

Each row represents one page/content item with its observed search and engagement measurements over the available 90-day window.

The model or scoring system will assign an opportunity score to each content item, and the resulting scores will be used to create a ranked review queue.

In [7]:
unit_cols = [
    "content_id",
    "client_id",
    "content_type",
    "main_intent",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "content_age_days"
]

unit_df = df[unit_cols].copy()

print(f"Rows: {len(unit_df):,}")
print(f"Columns: {len(unit_df.columns)}")
print("Unit of analysis: one content item")

unit_df.head(10)

Rows: 30,000
Columns: 10
Unit of analysis: one content item


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,content_age_days
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,0.76,10.6,5.88,187
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,0.05,20.3,0.00,445
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,0.09,36.5,0.00,141
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,0.49,6.2,1.28,463
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,0.13,44.0,0.00,263
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,0.03,8.5,0.00,147
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,0.00,7.0,0.00,90
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,0.06,21.2,3.57,445
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,32574,29,0.09,46.0,5.88,90
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,1240,2,0.16,4.9,0.00,257


## 5. Why ML beats a fixed rule here

A fixed rule such as "prioritize pages with low CTR" is useful as a baseline, but it treats pages too similarly.

CTR and engagement can vary with factors such as search position, impression volume, content type, intent, content age, and other measurable signals.

A scoring model can combine these signals and learn patterns that are difficult to express as a small set of manually chosen thresholds.

ML earns its place only if it can produce a more useful ranked queue than the simple rule-based baseline. If it does not improve the ranking quality, the simpler rule should be preferred.

In [8]:
baseline = df[df["impressions_90d"] >= 100].copy()

baseline["rule_score"] = 1 / (baseline["ctr"] + 1e-6)

print(f"Baseline-eligible pages: {len(baseline):,}")
print("Baseline: prioritize sufficiently visible pages with lower observed CTR.")

Baseline-eligible pages: 22,006
Baseline: prioritize sufficiently visible pages with lower observed CTR.


## 6. Self-check

- [x] Task type identified: Ranking / Scoring
- [x] Target/proxy identified as a future observed engagement outcome
- [x] Success metric identified: Precision@K
- [x] Unit of analysis shown as a real dataframe
- [x] Output tied to a real content-review action
- [x] Fixed rule baseline identified
- [x] ML is justified only if it improves on the baseline
- [X] Notebook runs top to bottom with no errors
- [X] Committed to my repo under `work/notebooks/`